# VQCC v15 - best-of public portfolio, every member re-verified

Per task, the cheapest circuit that passes the unmodified official verifier (local salt `cw1030a`, 2026-10-30):

| task | source | cost | ratio |
|---|---|---:|---:|
| invsqrt | Ahmed Berat Ozer, *DF-THC Group Compression* output (62 qubits, sha 6dbb98b3) | 100,940 | 20.574 |
| dqi | 889-qubit decoder (sha 7a378f47; somejoint / alexchilton lineage, shipped in the same output) | 4.0840e11 | 59.84 |
| dfthcsos | heliosli *The dataset is public now* 10-30 release (lambda_eff 48.324, T 16266, Q 430) | 3.37994e8 | 9.928 |

Projected geometric mean **23.035**. CPU only, no GPU.
Credits: Ahmed Berat Ozer, heliosli, yoyobar, busyaprime, mrmeows, somejoint, alexchilton, Dmitrii Gluzdov (all Apache-2.0 public work).

In [ ]:
import glob, hashlib, json, math, os, shutil, sys, zipfile, pathlib
COMP = next(p for p in ['/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation','/kaggle/input/google-verifiable-quantum-circuit-compilation'] if os.path.isdir(p))
def find(name, must):
    hits = [h for h in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if must in h]
    assert hits, name; return hits[0]
OZER = os.path.dirname(find('submission.zip', 'group-compression'))
HELI = os.path.dirname(find('submission.zip', 'dataset-is-public'))
WORK = pathlib.Path('/kaggle/working'); SUB = WORK/'sub'
shutil.rmtree(SUB, ignore_errors=True); SUB.mkdir()
with zipfile.ZipFile(f'{OZER}/submission.zip') as z:
    z.extract('invsqrt.kmb', SUB); z.extract('dqi.kmb', SUB)
with zipfile.ZipFile(f'{HELI}/submission.zip') as z:
    for n in z.namelist():
        if n.startswith('dfthcsos/'): z.extract(n, SUB)
EXPECT = {'invsqrt.kmb':'6dbb98b34c3b16f1c4e0', 'dqi.kmb':'7a378f4755605e0c5fa6', 'dfthcsos/submission.npz':'6503daaba4b703c74e44'}
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda: f.read(1<<22), b''): h.update(c)
    return h.hexdigest()
for k,v in EXPECT.items():
    assert sha(SUB/k).startswith(v), k
print('members match pinned hashes')

In [ ]:
sys.path[:0] = [f'{COMP}/kickmix', f'{COMP}/verifiers']
import kickmix as km
from invsqrt import verify_submission_invsqrt as vinv
from dqi import verify_submission_dqi as vdqi
from dfthcsos import verify as vdft
SALT = 'v15-commit'
QUICK = os.environ.get('KAGGLE_KERNEL_RUN_TYPE','') != 'Batch'  # interactive: quick check
spec = vinv.ProblemSpecInvSqrt.for_n_p(10)
if QUICK: spec = __import__('dataclasses').replace(spec, shots_per_x=1)
r_inv = vinv.verify(km.Circuit.from_file(str(SUB/'invsqrt.kmb'), format='kmb'), spec, salt=SALT, batch_size=4 if QUICK else 256)
r_dqi = vdqi.verify(km.Circuit.from_file(str(SUB/'dqi.kmb'), format='kmb'), vdqi.ProblemSpecDQI.benchmark_instance(), salt=SALT)
r_ch = vdft.verify(str(SUB/'dfthcsos'), salt=SALT)
REF = {'invsqrt': 102349*20.291, 'dqi': 4.1294e11*59.18, 'dfthcsos': 3.40572e8*9.853}  # calibrated on our scored v14
ratios = []
for t, r in (('invsqrt', r_inv), ('dqi', r_dqi), ('dfthcsos', r_ch)):
    assert r['status'] == 'PASS', (t, r)
    ratios.append(REF[t]/float(r['score'])); print(t, 'PASS', float(r['score']), round(ratios[-1], 4))
print('projected GM', round(math.prod(ratios)**(1/3), 4))

In [ ]:
with zipfile.ZipFile(WORK/'submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob('*')):
        if p.is_file(): z.write(p, p.relative_to(SUB).as_posix())
shutil.rmtree(SUB); print('wrote submission.zip')